# 🚀 AIC-2026: Fast DAM & Audio ASR Embedding on Kaggle GPU

This notebook uses **NVIDIA GPU (CUDA FP16)** to embed:
1. **DAM Object Captions** (435,713 objects with bounding boxes) $\rightarrow$ **`BAAI/bge-m3`** (1024-d Dense).
2. **Audio ASR Speech Transcripts** (55,168 segments) $\rightarrow$ **`BAAI/bge-m3`** (1024-d Dense).

*(Note: SigLIP-2 visual feature embeddings are already extracted as `.f16.npy` files and excluded from this notebook).*

Estimated total execution time: **~10–12 minutes** on a single Kaggle T4 / P100 GPU.

## 📦 Step 1: Install Dependencies & Check GPU

In [ ]:
!nvidia-smi
!pip install -q qdrant-client transformers torch tqdm accelerate
!apt-get update -qq && apt-get install -y -qq rclone zip unzip

## 🔑 Step 2: Configure Rclone for Google Drive Access

Paste your `rclone.conf` contents below or configure your Google Drive remote named `AIC_Drive`:

In [ ]:
import os
from pathlib import Path

# Ensure rclone config directory exists
rclone_dir = Path.home() / ".config" / "rclone"
rclone_dir.mkdir(parents=True, exist_ok=True)
rclone_conf = rclone_dir / "rclone.conf"

# If you have your rclone.conf contents, paste them here:
RCLONE_CONFIG_CONTENT = """
[AIC_Drive]
type = drive
scope = drive
token = YOUR_GOOGLE_DRIVE_RCLONE_TOKEN_HERE
"""

if "YOUR_GOOGLE_DRIVE_RCLONE_TOKEN_HERE" not in RCLONE_CONFIG_CONTENT:
    rclone_conf.write_text(RCLONE_CONFIG_CONTENT.strip(), encoding="utf-8")
    print(f"✅ Rclone config saved to {rclone_conf}")
else:
    print("⚠️ Please paste your valid rclone token or run `rclone config` in terminal!")

# Test rclone connection
!rclone listremotes

## 📥 Step 3: Download Raw DAM & ASR Files from Google Drive

Sync the raw JSONL and map CSV files to the local fast NVMe SSD (`/kaggle/working/data/`).

In [ ]:
!mkdir -p /kaggle/working/data/dam_descriptions
!mkdir -p /kaggle/working/data/asr_segments
!mkdir -p /kaggle/working/data/map-keyframes

# Sync input files using multi-threaded rclone transfers
!rclone copy "AIC_Drive:AIC_HCM/artifacts/dam_descriptions" /kaggle/working/data/dam_descriptions --transfers 16 -P
!rclone copy "AIC_Drive:AIC_HCM/artifacts/asr_segments" /kaggle/working/data/asr_segments --transfers 16 -P
!rclone copy "AIC_Drive:AIC_HCM/map-keyframes" /kaggle/working/data/map-keyframes --transfers 16 -P

# Verify download counts
import glob
print(f"DAM JSONLs: {len(glob.glob('/kaggle/working/data/dam_descriptions/*.jsonl'))} files")
print(f"ASR Files:  {len(glob.glob('/kaggle/working/data/asr_segments/*.*'))} files")
print(f"Map CSVs:   {len(glob.glob('/kaggle/working/data/map-keyframes/*.csv'))} files")

## ⚡ Step 4: Run High-Speed CUDA FP16 Embedding (BGE-M3)

In [ ]:
# Download the embedding runner script if not present locally
# Or execute the embedded script directly
!python -m run_kaggle_embedding \
    --dam-dir /kaggle/working/data/dam_descriptions \
    --asr-dir /kaggle/working/data/asr_segments \
    --map-dir /kaggle/working/data/map-keyframes \
    --output-db /kaggle/working/qdrant_db \
    --batch-size 128 \
    --model-id BAAI/bge-m3

## 📊 Step 5: Verify Qdrant Database Counts

In [ ]:
from qdrant_client import QdrantClient

client = QdrantClient(path="/kaggle/working/qdrant_db")
print("=== QDRANT EMBEDDED DATABASE SUMMARY ===")
for c in client.get_collections().collections:
    info = client.get_collection(c.name)
    print(f"Collection [{c.name}]: {info.points_count:,} points on disk")

## 📦 Step 6: Compress & Upload `qdrant_db.zip` Back to Google Drive

In [ ]:
# Compress the indexed Qdrant database
%cd /kaggle/working
!zip -q -r qdrant_db.zip qdrant_db/

import os
zip_size_mb = os.path.getsize("/kaggle/working/qdrant_db.zip") / (1024 * 1024)
print(f"📦 qdrant_db.zip size: {zip_size_mb:.2f} MB")

# Upload directly to Google Drive
!rclone copy /kaggle/working/qdrant_db.zip "AIC_Drive:AIC_HCM/" -P

print("\n🎉 UPLOAD COMPLETE! The full embedded database is safely saved on Google Drive.")